# ShapeAudit — auditoria entre duas versões de um modelo

Este notebook roda do começo ao fim no Colab e termina com um **laudo HTML**
aparecendo aqui dentro.

Use **Ambiente de execução → Executar tudo**. Leva cerca de dois minutos.


## 1. Instalar

In [ ]:
!pip install -q shapeaudit openpyxl

import shapeaudit
print('versão instalada:', shapeaudit.__version__)

## 2. Os dados

Usamos um conjunto real: **churn de telecom** (7.043 clientes, 13 variáveis,
alvo `churn`). Para usar a sua planilha, suba o arquivo pelo painel da
esquerda e troque o caminho abaixo.

In [ ]:
import pandas as pd

URL = ('https://raw.githubusercontent.com/IBM/'
       'telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv')
d = pd.read_csv(URL)

# ── para usar SUA planilha, comente as duas linhas acima e use: ──
# d = pd.read_excel('meus_dados.xlsx')

ALVO = 'Churn'
d = d.drop(columns=['customerID'])
d['TotalCharges'] = pd.to_numeric(d.TotalCharges, errors='coerce').fillna(0)
print(d.shape)
d.head(3)

## 3. Preparar e treinar duas versões

Aqui as duas versões são dois algoritmos diferentes. **No seu caso, use os
seus dois modelos já treinados** — o modelo que está em produção e o
candidato.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split

y = (d.pop(ALVO) == 'Yes').astype(int)
for c in d.columns:
    if not pd.api.types.is_numeric_dtype(d[c]):
        d[c] = pd.factorize(d[c])[0]
X = d.astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.4, random_state=42)

v1 = RandomForestClassifier(n_estimators=100, max_depth=6,
                            random_state=0, n_jobs=1).fit(X_tr, y_tr)
v2 = GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                random_state=0).fit(X_tr, y_tr)

print(f'versão 1: {v1.score(X_te, y_te):.3f}')
print(f'versão 2: {v2.score(X_te, y_te):.3f}')
print(f'diferença: {abs(v1.score(X_te, y_te) - v2.score(X_te, y_te)):.4f}')

As duas têm praticamente a mesma acurácia. **Qualquer revisão as aprovaria
como equivalentes.**

## 4. Auditar

In [ ]:
from shapeaudit import Guard

r = Guard().audit(v1, v2, X_te, y_te, list(X.columns), X_train=X_tr)

print('VEREDITO:', r.verdict, '  (decidido por:', r.verdict_driver + ')')
print('causa raiz:', r.root_cause_features or '—')

## 5. O que mudou de peso e de comportamento

In [ ]:
diff = pd.DataFrame(r.model_diff)
diff = diff[diff.categoria != 'estavel'].copy()
diff['influência'] = (diff.peso_v1.map('{:.0%}'.format) + ' → '
                      + diff.peso_v2.map('{:.0%}'.format))
diff['Δ'] = diff.delta_pp.map('{:+.1f} pp'.format)
diff[['feature', 'influência', 'Δ', 'categoria', 'forma_mudou']]

A linha com `peso_estavel_forma_mudou` é a que interessa: **olhando só
importância de variáveis, ninguém veria que ela mudou.**

## 6. O laudo, aqui dentro

In [ ]:
from IPython.display import HTML, display

html = r.to_html(lang='pt')
display(HTML(html))

## 7. Salvar o laudo

In [ ]:
with open('laudo.html', 'w', encoding='utf-8') as f:
    f.write(html)

from google.colab import files      # comente esta célula fora do Colab
files.download('laudo.html')

## 8. Medir o impacto na sua decisão

O laudo pergunta *"isso muda decisões?"*. Para responder, informe a sua
regra — aqui, selecionar os 10% de maior risco.

In [ ]:
import numpy as np

k = int(.10 * len(X_te))
top1 = set(np.argsort(-v1.predict_proba(X_te)[:, 1])[:k])
top2 = set(np.argsort(-v2.predict_proba(X_te)[:, 1])[:k])
trocados = len(top1 - top2)

print(f'{trocados} de {k} clientes selecionados mudaram '
      f'({100 * trocados / k:.1f}%)')

display(HTML(r.to_html(lang='pt', decisao={
    'trocados': trocados, 'selecionados': k,
    'pct': round(100 * trocados / k, 1)})))

## 9. Calibrar o limiar no seu domínio

O limiar padrão vem de uma bancada sintética. Calibre com retreinos que você
**teria promovido** — não precisa de rótulo nem de falha conhecida.

In [ ]:
from shapeaudit.frechet import calibrar_limiar

pares = [(RandomForestClassifier(n_estimators=100, max_depth=6,
                                 random_state=s, n_jobs=1).fit(X_tr, y_tr),
          RandomForestClassifier(n_estimators=100, max_depth=6,
                                 random_state=s + 50, n_jobs=1).fit(X_tr, y_tr))
         for s in range(4)]

print(calibrar_limiar(pares, X_te, list(X.columns)))

## 10. E quando algo quebra de verdade?

Até aqui as duas versões eram saudáveis, só treinadas com algoritmos
diferentes. Agora o caso que todo mundo teme: **uma coluna chega constante**
depois de uma mudança no pipeline de dados.

A acurácia quase não se mexe — é isso que torna o caso perigoso.

In [ ]:
X_quebrado = X_tr.copy()
X_quebrado['Contract'] = X_quebrado['Contract'].mean()   # a coluna virou constante

v2_quebrado = RandomForestClassifier(n_estimators=100, max_depth=6,
                                     random_state=0, n_jobs=1).fit(X_quebrado, y_tr)

print(f'versão boa:      {v1.score(X_te, y_te):.3f}')
print(f'versão quebrada: {v2_quebrado.score(X_te, y_te):.3f}')
print(f'diferença:       {abs(v1.score(X_te, y_te) - v2_quebrado.score(X_te, y_te)):.4f}')

Repare: a diferença de acurácia é pequena. **Um gate por métrica aprovaria
esta versão.**

In [ ]:
r_quebrado = Guard().audit(v1, v2_quebrado, X_te, y_te, list(X.columns),
                          X_train=X_quebrado)

print('VEREDITO:', r_quebrado.verdict)
print('comece por:', r_quebrado.root_cause_features)

display(HTML(r_quebrado.to_html(lang='pt')))

O laudo agora diz **Não promova**, nomeia a variável e explica o que
investigar primeiro. Compare com o laudo da seção 6: mesma ferramenta,
diagnósticos diferentes.

**Os três resultados possíveis, nesta ordem de gravidade:**

| veredito | o que significa |
|---|---|
| Não promova | uma variável sumiu do modelo — conserte o pipeline |
| Revise antes de promover | o comportamento mudou com os mesmos dados |
| Pode promover | nenhuma camada encontrou mudança |

## O que ler antes de confiar

- degradação parcial não bloqueia até 80% das linhas de uma variável;
- em conjuntos de avaliação pequenos (menos de ~400 linhas) a camada de forma
  pede revisão em cerca de um terço dos retreinos saudáveis;
- fora de árvores (MLP, ridge) é experimental;
- a ferramenta diz que as duas versões discordam, **não** qual está certa.
